
### 1. Clean Transaction Data
*Cast data types and engineer the `shelf_price` feature.*

In [0]:
from pyspark.sql.functions import col, round, when, trim, coalesce, lit, upper,  regexp_replace

In [0]:
# Read from Bronze
df_transactions = spark.read.table("dunnhumby_project.bronze.transaction_data")

# Cast currency columns first so arithmetic uses exact decimal math
df_casted = df_transactions.select(
    # Keys
    col("basket_id").cast("long").alias("basket_id"),
    col("household_key").cast("int").alias("household_key"),
    col("product_id").cast("int").alias("product_id"),
    col("store_id").cast("int").alias("store_id"),
    # Time Dimensions
    col("week_no").cast("int").alias("week_no"),
    col("day").cast("int").alias("day"),
    col("trans_time").cast("int").alias("trans_time"),
    # Quantities & Currency Metrics
    col("quantity").cast("int").alias("quantity"),
    col("sales_value").cast("decimal(10, 2)").alias("sales_value"),
    col("retail_disc").cast("decimal(10, 2)").alias("retail_disc"),
    col("coupon_disc").cast("decimal(10, 2)").alias("coupon_disc"),
    col("coupon_match_disc")
    .cast("decimal(10, 2)")
    .alias("coupon_match_disc"),
)

# Calculate shelf price
df_transactions_clean = df_casted.withColumn(
    "shelf_price",
    when(
        col("quantity") > 0,
        round(
            (
                col("sales_value")
                - (col("retail_disc") + col("coupon_match_disc"))
            )
            / col("quantity"),
            2,
        ),
    )
    .otherwise(lit(0))
    .cast("decimal(10, 2)"),
)

print("Transaction table cleaned successfully!")

Transaction table cleaned successfully!


### 2. Clean Product Table

*Handle missing departments and sizes identified in EDA.*

In [0]:
df_product = spark.read.table("dunnhumby_project.bronze.product")

df_product_clean = df_product.select(
    # Primary Key
    col("product_id").cast("int").alias("product_id"),
    # Manufactuer
    col("manufacturer").cast("int").alias("manufacturer"),
    upper(trim(coalesce(col("brand"), lit("UNKNOWN")))).alias("brand"),
    # Product Hierarchy (Broad -> Specific)
    upper(trim(coalesce(col("department"), lit("UNKNOWN")))).alias("department"),
    upper(trim(coalesce(col("commodity_desc"), lit("UNKNOWN")))).alias(
        "commodity_desc"
    ),
    upper(trim(coalesce(col("sub_commodity_desc"), lit("UNKNOWN")))).alias(
        "sub_commodity_desc"
    ),
    # Product Size
    when(
        col("curr_size_of_product").isNull()
        | (regexp_replace(col("curr_size_of_product"), r"\s+", "") == ""),
        lit("UNKNOWN"),
    )
    .otherwise(upper(trim(col("curr_size_of_product"))))
    .alias("curr_size_of_product")
)


print("Product table cleaned!")


Product table cleaned!


### 3. Clean Household Demographics Table
*Ensure correct data types and standardize text.*


In [0]:
df_household = spark.read.table("dunnhumby_project.bronze.hh_demographic")

df_household_clean = df_household.select(
    # Key
    col("household_key").cast("int").alias("household_key"),
    # Class
    upper(trim(col("classification_1"))).alias("classification_1"),
    upper(trim(col("classification_2"))).alias("classification_2"),
    upper(trim(col("classification_3"))).alias("classification_3"),
    upper(trim(col("classification_5"))).alias("classification_5"),
    upper(trim(col("classification_4"))).alias("classification_4"),
    # Description
    upper(trim(col("homeowner_desc"))).alias("homeowner_desc"),
    upper(trim(col("kid_category_desc"))).alias("kid_category_desc")
)

print("Household table cleaned!")

Household table cleaned!


### 4. Clean the Campaign Tables
We will combine the campaign description with the campaign table and ensure dates are correct.

In [0]:
df_campaign = spark.read.table("dunnhumby_project.bronze.campaign_table")
df_campaign_desc = spark.read.table("dunnhumby_project.bronze.campaign_desc")

df_campaign_cleaned = df_campaign.select(
    col("household_key").cast("int").alias("household_key"),
    col("campaign").cast("int").alias("campaign"),
    upper(trim(col("description"))).alias("description")
)

df_campaign_desc_cleaned = df_campaign_desc.select(
    col("campaign").cast("int").alias("campaign"),
    upper(trim(col("description"))).alias("description"),
    col("start_day").cast("int").alias("start_day"),
    col("end_day").cast("int").alias("end_day")
)

df_campaign_cleaned.write.format("delta").mode("overwrite").saveAsTable("dunnhumby_project.silver.campaign")
df_campaign_desc_cleaned.write.format("delta").mode("overwrite").saveAsTable("dunnhumby_project.silver.campaign_desc")
print("Campaign table cleaned and saved")

Campaign table cleaned and saved


## 5. Clean the Causal Data and Coupons
Ensure the data types are correct before joining.

In [0]:
df_causal = spark.read.table("dunnhumby_project.bronze.causal_data")
df_coupon_redempt = spark.read.table("dunnhumby_project.bronze.coupon_redempt")
df_coupon = spark.read.table("dunnhumby_project.bronze.coupon")

# Causal Data
df_causal_clean = df_causal.select(
    col("product_id").cast("int").alias("product_id"),
    col("store_id").cast("int").alias("store_id"),
    col("week_no").cast("int").alias("week_no"),
    col("display").alias("display"),
    col("mailer").alias("mailer")
)
   
# Coupon Redemptions
df_coupon_redempt_clean = df_coupon_redempt.select(
    col("coupon_upc").cast("string").alias("coupon_upc"),
    col("household_key").cast("int").alias("household_key"),
    col("campaign").cast("int").alias("campaign"),
    col("day").cast("int").alias("day")
)

df_coupon_clean = df_coupon.select(
    col("coupon_upc").cast("string").alias("coupon_upc"),
    col("product_id").cast("int").alias("product_id"),
    col("campaign").cast("int").alias("campaign")
)

df_causal_clean.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable("dunnhumby_project.silver.causal")
df_coupon_redempt_clean.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable("dunnhumby_project.silver.coupon_redempt")
df_coupon_clean.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable("dunnhumby_project.silver.coupon")


print("Causal and Coupon tables cleaned and saved")

Causal and Coupon tables cleaned and saved


### 6. Create the Master Silver Table
*Join all cleaned tables into one analytical table for the Gold Layer.*


In [0]:

df_silver = df_transactions_clean.join(df_product_clean, on="product_id", how="left")

# Join demographics (Left join because only 801 households have demographic data)
df_silver = df_silver.join(df_household_clean, on="household_key", how="left")

# Fill null demographics with 'UNKNOWN' for the households that didn't have demographic data
demo_cols = ["classification_1", "classification_2", "classification_3", "classification_4", "classification_5", "homeowner_desc", "kid_category_desc"]
for c in demo_cols:
    df_silver = df_silver.withColumn(c, coalesce(col(c), lit("UNKNOWN")))

df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dunnhumby_project.silver.transactions_enriched")

print(f"Master Silver table created successfully! Total rows: {df_silver.count()}")

Master Silver table created successfully! Total rows: 2595732


In [0]:
df_silver.limit(5).display()

household_key,product_id,basket_id,store_id,week_no,day,trans_time,quantity,sales_value,retail_disc,coupon_disc,coupon_match_disc,shelf_price,manufacturer,brand,department,commodity_desc,sub_commodity_desc,curr_size_of_product,classification_1,classification_2,classification_3,classification_5,classification_4,homeowner_desc,kid_category_desc
1134,842126,32556796222,356,54,375,1239,2,0.84,0.00,0.00,0.00,0.42,1282,NATIONAL,DRUG GM,BABY FOODS,BABY FOOD - BEGINNER,4 OZ,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN
2375,1082185,26984851472,364,1,1,1631,1,1.21,0.00,0.00,0.00,1.21,2,NATIONAL,PRODUCE,TROPICAL FRUIT,BANANAS,40 LB,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN
2375,1036325,26984851472,364,1,1,1631,1,0.99,-0.30,0.00,0.00,1.29,69,PRIVATE,PRODUCE,VEGETABLES - ALL OTHERS,CELERY,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN
2375,1004906,26984851472,364,1,1,1631,1,1.39,-0.60,0.00,0.00,1.99,69,PRIVATE,PRODUCE,POTATOES,POTATOES RUSSET (BULK&BAG),5 LB,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN
2375,8160430,26984851472,364,1,1,1631,1,1.50,-0.39,0.00,0.00,1.89,69,PRIVATE,PRODUCE,ORGANICS FRUIT & VEGETABLES,ORGANIC CARROTS,1 LB,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN
